<a href="https://colab.research.google.com/github/KavinMadusanka/fake-news-detection/blob/feat%2Fpreprocess/notebooks/data_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Step 1: Basic GitHub Repository Setup**
This section connects Google Colab with your GitHub repository.


*   It defines key variables such as repository name, username, token, and branch.
*   git config sets your Git identity for commits
*   The repository is cloned using a personal access token.
*   Finally, a new feature branch named feat/preprocess is created and checked out.


This setup ensures that all data preprocessing work is version-controlled and pushed to the correct branch.


In [1]:
# === BASIC SETUP (run once per session) ===
REPO = "fake-news-detection"
USER_OR_ORG = "KavinMadusanka"
TOKEN = "token"  # paste at runtime ONLY
BRANCH = "feat/preprocess" # your branch name

!git config --global user.email "kavinmadusanka20011@gmail.com"
!git config --global user.name "KavinMadusanka"

!git clone https://{TOKEN}@github.com/{USER_OR_ORG}/{REPO}.git
%cd /content/{REPO}
!git checkout -b {BRANCH}


Cloning into 'fake-news-detection'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 167 (delta 78), reused 55 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 2.84 MiB | 17.34 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/fake-news-detection
Switched to a new branch 'feat/preprocess'


### **Step 2: Upload Dataset Files**
This allows manual uploading of dataset files (e.g., Fake.csv and True.csv) from your local system into the Colab environment.

files.upload() opens a file picker and returns a dictionary of uploaded files.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Fake.csv to Fake.csv
Saving True.csv to True.csv


In [3]:
import pandas as pd

fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

print("Fake:", fake.shape)
print("True:", true.shape)
print(fake.columns.tolist())   # should show ['title','text','subject','date']

Fake: (23481, 4)
True: (21417, 4)
['title', 'text', 'subject', 'date']


In [4]:
fake['label'] = 1   # 1 = fake
true['label'] = 0   # 0 = real

df = pd.concat([
    fake[['title','text','subject','date','label']],
    true[['title','text','subject','date','label']]
], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

# Combine title + text into one column for NLP
df['text'] = (df['title'].fillna('') + ' ' + df['text'].fillna('') + ' ' + df['subject'].fillna('') + ' ' + df['date'].fillna('')).str.strip()
df = df[['text','subject', 'date','label']]
display(df.head())

,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,US_News,"February 13, 2017",1
1,Trump drops Steve Bannon from National Securit...,politicsNews,"April 5, 2017",0
2,Puerto Rico expects U.S. to lift Jones Act shi...,politicsNews,"September 27, 2017",0
3,OOPS: Trump Just Accidentally Confirmed He Lea...,News,"May 22, 2017",1
4,Donald Trump heads for Scotland to reopen a go...,politicsNews,"June 24, 2016",0


In [5]:
df.to_csv('fake-news-dataset.csv', index=False)
print(" Saved fake-news-dataset.csv")

 Saved fake-news-dataset.csv


In [6]:
!pip install emoji unidecode nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.9 MB/s eta 0:00:00


In [7]:
# --- CLEANING ---
import re, emoji
from unidecode import unidecode
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

def clean_text(s):
    if pd.isna(s): return ""
    s = unidecode(str(s))
    s = emoji.replace_emoji(s, replace=' ')
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return " ".join(w for w in s.split() if w not in STOPWORDS)

df['text_clean'] = df['text'].apply(clean_text)
df = df[df['text_clean'].str.len() > 0]
df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,subject,date,label,text_clean
0,Ben Stein Calls Out 9th Circuit Court: Committ...,US_News,"February 13, 2017",1,ben stein calls 9th circuit court committed co...
1,Trump drops Steve Bannon from National Securit...,politicsNews,"April 5, 2017",0,trump drops steve bannon national security cou...
2,Puerto Rico expects U.S. to lift Jones Act shi...,politicsNews,"September 27, 2017",0,puerto rico expects u lift jones act shipping ...
3,OOPS: Trump Just Accidentally Confirmed He Lea...,News,"May 22, 2017",1,oops trump accidentally confirmed leaked israe...
4,Donald Trump heads for Scotland to reopen a go...,politicsNews,"June 24, 2016",0,donald trump heads scotland reopen golf resort...


In [8]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import numpy as np, os

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Optional oversample
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(train_df[['text_clean']], train_df['label'])
train_df = pd.concat([X_res, y_res], axis=1)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

Train: (37570, 2) Val: (4490, 5) Test: (4490, 5)


In [9]:
os.makedirs('data/processed', exist_ok=True)

train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)

print("Saved train/val/test in data/processed/")

Saved train/val/test in data/processed/


In [10]:
%%bash
mkdir -p notebooks data/raw data/processed results/figs report src

# create .gitignore
cat > .gitignore <<'EOF'
data/*
!data/README.md
results/*
!results/.gitkeep
*.csv
*.tsv
*.jsonl
*.joblib
*.pt
*.h5
*.keras
.kaggle/
.ipynb_checkpoints/
__pycache__/
EOF

# minimal placeholders
echo "Raw & processed data live here but are ignored by git." > data/README.md
touch results/.gitkeep
echo "## Fake News Detection" > README.md
echo "pandas
numpy
scikit-learn
imbalanced-learn
nltk
emoji
unidecode
matplotlib" > requirements.txt

In [11]:
!git add .gitignore data/README.md results/.gitkeep README.md requirements.txt
!git commit -m "Initial repo structure: notebooks/, data/, results/, .gitignore, README, requirements"
!git push --set-upstream origin feat/preprocess

On branch feat/preprocess
nothing to commit, working tree clean
fatal: could not read Password for 'https://token@github.com': No such device or address


In [12]:
from google.colab import files
files.download("data/processed/train.csv")
files.download("data/processed/val.csv")
files.download("data/processed/test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
!git add notebooks/data_preprocess.ipynb
!git commit -m "preprocessing complete"
!git push --set-upstream origin feat/preprocess

On branch feat/preprocess
nothing to commit, working tree clean
fatal: could not read Password for 'https://token@github.com': No such device or address
